In [ ]:
import os
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import shap
from matplotlib.colors import LinearSegmentedColormap

# 从 CSV 和 JSON 文件加载变量（更稳健的方法）
feature_importance_df = pd.read_csv("feature_importance.csv")

with open("plot_variables.json", 'r') as f:
    saved_vars = json.load(f)

# 提取变量，并在需要时将数据转回 NumPy 数组
features = saved_vars['features']
importances = np.array(saved_vars['importances'])
avg_meta_shap = np.array(saved_vars['avg_meta_shap'])
avg_meta_features = np.array(saved_vars['avg_meta_features'])
base_model_names = saved_vars['base_model_names']




import matplotlib.pyplot as plt
import numpy as np
import shap

# 设置字体为 Times New Roman
plt.rcParams.update({'font.family': 'Times New Roman'})

# # 提取特征名和重要性值
# features = feature_importance_df['Feature']
# importances = feature_importance_df['Importance']

# 原始颜色列表
colors_original = ['#FDD379', '#F7C2CD', '#A6DAEF', '#B0D9A5', '#E68D3D', '#E26472', '#6270B7', '#077535']

# 新增颜色
colors_new = ['#8E44AD','#FFD700',  '#3498DB', '#16A085', '#D35400', '#C0392B', '#7F8C8D', '#2ECC71']

# 合并原始颜色和新增颜色
colors = colors_original + colors_new
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
# 确保颜色数量与特征数量一致（截取匹配数量）
colors = colors[:len(features)]

# 创建一行两列的子图布局
fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={'width_ratios': [1.6, 1]})

# 创建一行两列的子图布局
fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={'width_ratios': [1.6, 1]})
plt.sca(axes[0])  # 设置当前子图为左侧

# 绘制左侧 SHAP 蜂巢图

start_color = (228/255, 107/255, 144/255)  # 修正超出范围的值
end_color = (130/255, 170/255, 231/255)

# 创建渐变色映射
custom_cmap = LinearSegmentedColormap.from_list("custom_pink_blue", 
                                              [start_color, (1, 1, 1), end_color])


# # 设置起始和结束颜色
# start_color = (228/255, 107/255, 144/255)  # 粉红色
# end_color = (130/255, 170/255, 231/255)    # 蓝色

# # 计算一个偏紫色的中间色调（两种颜色的平均值）
# mid_color = ((start_color[0] + end_color[0])/2, 
#              (start_color[1] + end_color[1])/2,
#              (start_color[2] + end_color[2])/2)

# # 创建渐变色映射 - 使用计算出的中间色
# custom_cmap = LinearSegmentedColormap.from_list("custom_pink_blue", 
#                                               [start_color, mid_color, end_color],
#                                               N=256)



shap.summary_plot(avg_meta_shap, avg_meta_features, feature_names=base_model_names, plot_type="dot", show=False,cmap=custom_cmap)
# plt.cm.RdBu_r
axes[0].set_title('SHAP Summary Plot', fontsize=17)
# 调整图 a 的字体大小和布局
plt.tick_params(axis='x', labelsize=15)  # 增加x轴刻度标签大小
plt.tick_params(axis='y', labelsize=15)  # 增加y轴刻度标签大小
cbar = plt.gcf().axes[-1]  # 获取colorbar轴（假设是最后一个轴）
cbar.tick_params(labelsize=15)  # 调整colorbar刻度标签大小
cbar.set_ylabel("Feature Value", fontsize=15, labelpad=-3, family="Times New Roman")  # 设置colorbar标签的字体和移动位置
cbar.yaxis.set_label_coords(3, 0.5)  # 移动标签到左边
plt.xlabel("Attributions value", fontsize=15, family="Times New Roman")

# # 设置子图位置和大小 - 向左移动环形图
# # [left, bottom, width, height] - left值减小会使图向左移动
# axes[1].set_position([0.5, 0.1, 0.4, 0.8])  # 调整left值（第一个参数）使其更小
# 绘制右侧环形图
plt.sca(axes[1])  # 设置当前子图为右侧
wedges, texts = axes[1].pie(
    importances,
    startangle=90,
    radius=1.5,
    wedgeprops=dict(width=0.7, edgecolor='white', linewidth=2),  # 设置间隔
    colors=colors,  # 使用扩展后的颜色
    labels=None  # 不在区域中显示标签
)

# 添加百分比到区域外部
for i, (wedge, importance) in enumerate(zip(wedges, importances)):
    angle = (wedge.theta2 - wedge.theta1) / 2 + wedge.theta1  # 计算区域角度
    x = 1.2 * wedge.r * np.cos(np.radians(angle))  # 调整百分比位置 (x 坐标)
    y = 1.2 * wedge.r * np.sin(np.radians(angle))  # 调整百分比位置 (y 坐标)
    
    percentage = importance / sum(importances)
    
    if percentage < 0.03:  # 针对小于4.6%的值，不显示
        # 不显示任何内容
        pass
    elif percentage < 0.01:  # 针对小于1%但大于等于0.046%的值，用连线标注
        # 外部连接点
        x_annotate = 1 * wedge.r * np.cos(np.radians(angle))
        y_annotate = 1.4 * wedge.r * np.sin(np.radians(angle))
        # 绘制连线
        axes[1].annotate(
            f'{percentage:.1%}', 
            xy=(x, y), 
            xytext=(x_annotate, y_annotate),
            arrowprops=dict(arrowstyle="-", color='black'),
            ha='center', 
            va='center', 
            fontsize=14
        )
    else:
        # 正常显示百分比
        axes[1].text(x, y, f'{percentage:.1%}', ha='center', va='center', fontsize=14)

# 添加环形图中心文本
axes[1].text(0, 0, 'metal model', ha='center', va='center', fontsize=16)

# 添加图例
axes[1].legend(
    wedges, features,
    title="Features",
    loc="center left",
    bbox_to_anchor=(1.45, 0.5),  # 图例位置，调整到右侧
    fontsize=14,
    title_fontsize=14,
)

axes[1].set_title('Feature Importance Donut Chart', fontsize=18, 
                  pad=0,             # 增加 pad 值，控制标题与图之间的距离
                  y=1.445,             # y 位置：1.0 是默认位置，>1 表示上移，<1 表示下移
                  x=0.65)              # x 位置：0.5 是居中，<0.5 向左移，>0.5 向右移

# 调整布局并保存
# plt.tight_layout()
plt.savefig("7.pdf", format='pdf', bbox_inches='tight')
plt.show()